In [ ]:
import torch

from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torchvision import transforms

import matplotlib.pyplot as plt

from torch import nn
from torchinfo import summary

from tqdm.auto import tqdm
from timeit import default_timer as timer

from torchmetrics import ConfusionMatrix
from mlxtend.plotting import plot_confusion_matrix


torch.use_deterministic_algorithms(True)

In [ ]:
train_directory = "dataset/divided/train/"
val_directory = "dataset/divided/val/"
test_directory = "dataset/divided/test/"

In [ ]:
transform_train = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((360, 360)),
    transforms.RandomRotation(
        degrees=40,
        fill=125
    ),
    transforms.RandomPerspective(
        distortion_scale=0.6,
        p=1.0,
        fill=125
    ),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.ColorJitter(brightness=0.3),
    transforms.ToTensor()
])

transform_test = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((360, 360)),
    transforms.ToTensor()
])

In [ ]:
train_dataset = ImageFolder(
    root=train_directory,
    transform=transform_train
)

val_dataset = ImageFolder(
    root=val_directory,
    transform=transform_test
)

test_dataset = ImageFolder(
    root=test_directory,
    transform=transform_test
)

In [ ]:
train_dataset = ImageFolder(train_directory, transform_train)
val_dataset = ImageFolder(val_directory, transform_test)
test_dataset = ImageFolder(test_directory, transform_test)

In [ ]:
imagen, etiqueta = train_dataset[50]

plt.imshow(
    imagen.permute(1, 2, 0),
    cmap="gray"
)

plt.title(
    f"Clase: {train_dataset.classes[etiqueta]}"
)

plt.axis("off")
plt.show()

In [ ]:
BATCH_SIZE = 64

train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_dataloader = DataLoader(
    dataset=val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_dataloader = DataLoader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

len(train_dataloader), len(val_dataloader), len(test_dataloader)

In [ ]:
class TinyVGG(nn.Module):

    def __init__(
        self,
        input_shape: int,
        hidden_units: int,
        output_shape: int
    ):
        super().__init__()

        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(
                in_channels=input_shape,
                out_channels=hidden_units,
                kernel_size=3,
                stride=1,
                padding=0
            ),
            nn.ReLU(),

            nn.Conv2d(
                in_channels=hidden_units,
                out_channels=hidden_units,
                kernel_size=3,
                stride=1,
                padding=0
            ),
            nn.ReLU(),

            nn.MaxPool2d(
                kernel_size=2,
                stride=2
            )
        )

        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(
                in_channels=hidden_units,
                out_channels=hidden_units,
                kernel_size=3,
                stride=1,
                padding=0
            ),
            nn.ReLU(),

            nn.Conv2d(
                in_channels=hidden_units,
                out_channels=hidden_units,
                kernel_size=3,
                stride=1,
                padding=0
            ),
            nn.ReLU(),

            nn.MaxPool2d(
                kernel_size=2,
                stride=2
            )
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),

            nn.Linear(
                in_features=hidden_units * 87 * 87,
                out_features=output_shape
            )
        )

    def forward(self, x):

        x = self.conv_block_1(x)
        # print(x.shape)

        x = self.conv_block_2(x)
        # print(x.shape)

        x = self.classifier(x)
        # print(x.shape)

        return x

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Dispositivo:", device)

torch.manual_seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

model_0 = TinyVGG(
    input_shape=1,
    hidden_units=8,
    output_shape=len(train_dataset.classes)
).to(device)

model_0

In [ ]:
image_batch, label_batch = next(
    iter(train_dataloader)
)

print("Forma de las imágenes:")
print(image_batch.shape)

print("Forma de las etiquetas:")
print(label_batch.shape)

In [ ]:
model_0(image_batch.to(device))

In [ ]:
summary(
    model_0,
    input_size=(1, 1, 360, 360)
)

In [ ]:
def train_step(model,
               dataloader,
               loss_fn,
               optimizer,
               device=device):
    model.train()
    train_loss, train_acc = 0, 0
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        y_pred = model(X)
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        y_pred_class = torch.argmax(y_pred, dim=1)
        train_acc += (y_pred_class == y).sum().item()/len(y_pred)
    train_loss = train_loss / len(dataloader)
    train_acc = train_acc / len(dataloader)
    return train_loss, train_acc

def val_step(model,
             dataloader,
             loss_fn,
             optimizer,
             device=device):
    model.eval ()
    val_loss, val_acc = 0, 0
    with torch.inference_mode():
        for batch, (X, y) in enumerate(dataloader):
            X, y = X.to(device), y.to(device)
            val_pred_logits = model(X)
            loss = loss_fn(val_pred_logits, y)
            val_loss += loss.item()
            y_pred_class = torch.argmax(val_pred_logits, dim=1)
            val_acc += (y_pred_class == y).sum().item()/len(val_pred_logits)
    val_loss = val_loss / len(dataloader)
    val_acc = val_acc / len(dataloader)
    return val_loss, val_acc

In [ ]:
def train(model,
          train_dataloader,
          val_dataloader,
          optimizer,
          loss_fn,
          epochs,
          device=device):
    results = {"train_loss": [],
               "train_acc": [],
               "val_loss": [],
               "val_acc": []}
    best_val_loss = torch.inf
    MODEL_PATH = 'model_trained/'
    for epoch in tqdm(range(epochs)):
        train_loss, train_acc = train_step(model=model,
                                           dataloader=train_dataloader,
                                           loss_fn=loss_fn,
                                           optimizer=optimizer,
                                           device=device)
        val_loss, val_acc = val_step(model=model,
                                     dataloader=val_dataloader,
                                     loss_fn=loss_fn,
                                     optimizer=optimizer,
                                     device=device)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            MODEL_NAME = "AngryHappySad.pth"
            MODEL_SAVE_PATH = MODEL_PATH + MODEL_NAME
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            
        print(f"Epoch: {epoch} | Train loss: {train_loss:.4f} Train acc: {train_acc:.4f} Val loss: {val_loss:.4f} Val acc: {val_acc:.4f}")
        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["val_loss"].append(val_loss)
        results["val_acc"].append(val_acc)
    return results

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

NUM_EPOCHS = 200
model_0 = TinyVGG(
    input_shape=1,
    hidden_units=10,
    output_shape=len(train_dataset.classes)
).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    params=model_0.parameters(),
    lr=0.001
)

In [ ]:
start_time = timer()
model_0_results = train(
    model=model_0,
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    epochs=NUM_EPOCHS
)
end_time = timer()
print(f"Tiempo de entrenamiento: {end_time - start_time:.3f} segundos")

In [ ]:
def plot_loss_curves(results):
    train_loss = results["train_loss"]
    test_loss = results["val_loss"]
    train_acc = results["train_acc"]
    test_acc = results["val_acc"]
    epochs = range(len(train_loss))
    
    plt.figure(figsize=(15, 7))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_loss, label="train_loss")
    plt.plot(epochs, test_loss, label="test_loss")
    plt.title("Loss")
    plt.xlabel("Epochs")
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(epochs, train_acc, label="train_acc")
    plt.plot(epochs, test_acc, label="test_acc")
    plt.title("Accuracy")
    plt.xlabel("Epochs")
    plt.legend()

In [ ]:
plot_loss_curves(model_0_results)

In [ ]:
model_0 = TinyVGG(
    input_shape=1,
    hidden_units=10,
    output_shape=len(train_dataset.classes)
).to(device)
model_0
MODEL_PATH = 'model_trained/'
MODEL_NAME = "AngryHappySad.pth"
MODEL_SAVE_PATH = MODEL_PATH + MODEL_NAME
model = torch.load(MODEL_SAVE_PATH)
model_0.load_state_dict(torch.load(f=MODEL_SAVE_PATH))

In [ ]:
def make_predictions(model, dataloader, device):
    pred_class = []
    model.to(device)
    model.eval()
    with torch.inference_mode():
        for batch, (X, y) in enumerate(dataloader):
            X, y = X.to(device), y.to(device)
            pred_logits = model(X)
            pred_prob = torch.softmax(pred_logits, dim=1)
            pred_class.append(torch.argmax(pred_prob, dim=1))
    return torch.hstack(pred_class)

In [ ]:
pred_classes = make_predictions(
    model=model_0,
    dataloader=test_dataloader,
    device=device
)
pred_classes

In [ ]:
from torchmetrics import ConfusionMatrix
from mlxtend.plotting import plot_confusion_matrix
%matplotlib inline
confmat = ConfusionMatrix(num_classes= 3 , task="multiclass")
confmat_tensor = confmat(pred_classes.cpu(), torch.tensor(test_dataset.targets), dtype=torch.int64)

fig, ax = plot_confusion_matrix(
    conf_mat=confmat_tensor.numpy(),
    class_names=test_dataset.classes,
    figsize=(10, 7)
)

In [ ]:
from sklearn.metrics import classification_report
pred_cpu = pred_classes.cpu().numpy()
test_dataset.targets
print(classification_report(y_true=test_dataset.targets, y_pred=pred_cpu, target_names=test_dataset.classes))

In [ ]:
test_sample = []
test_label = []

for sample, label in test_dataset:
    test_sample.append(sample)
    test_label.append(label)

test_sample[0].shape

In [ ]:
plt.figure(figsize=(20, 20))

nrows = 15
ncols = 5

for i, sample in enumerate(test_dataset):

    plt.subplot(nrows, ncols, i + 1)

    plt.imshow(
        sample[0].permute(1, 2, 0),
        cmap="gray"
    )

    pred_label = test_dataset.classes[pred_classes[i]]
    truth_label = test_dataset.classes[test_labels[i]]

    title_text = (
        f"Pred: {pred_label} | Truth: {truth_label}"
    )

    # Check for equality
    if pred_label == truth_label:
        plt.title(
            title_text,
            fontsize=8,
            c="g"
        )
    else:
        plt.title(
            title_text,
            fontsize=8,
            c="r"
        )

    plt.axis(False)

In [ ]:
import cv2
import numpy as np

%matplotlib qt

cap = cv2.VideoCapture(0)

fig = plt.figure(figsize=(5, 5))
ax = fig.add_subplot(111)

if not cap.isOpened():
    print("Error: Couldn't open camera.")

else:
    while True:

        ret, frame = cap.read()

        if not ret:
            print("Error: frame not loaded.")
            break

        # Convertir el fotograma a escala de grises
        gray = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2GRAY
        )

        # Ajustar la imagen al tamaño requerido por el modelo
        resized = cv2.resize(
            gray,
            (360, 360)
        )

        # Convertir la imagen en tensor
        tensor = (
            transforms
            .ToTensor()(resized)
            .unsqueeze(0)
            .to(device)
        )

        # Realizar la predicción
        model_0.eval()

        with torch.inference_mode():
            pred_logits = model_0(tensor)

            pred_prob = torch.softmax(
                pred_logits,
                dim=1
            )

            pred_class = torch.argmax(
                pred_prob,
                dim=1
            ).item()

        # Obtener el nombre de la clase
        pred_label = test_dataset.classes[pred_class]

        # Actualizar la ventana
        ax.clear()

        ax.imshow(
            resized,
            cmap="gray"
        )

        ax.set_title(
            f"Predicted: {pred_label}"
        )

        ax.axis("off")

        plt.pause(0.1)

        # Terminar cuando se cierre la ventana
        if not plt.fignum_exists(fig.number):
            break

cap.release()